# Capítulo 1 — Fundamentos de Inteligência Artificial e Machine Learning

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Identificar** as três tradições históricas da IA — simbólica, conexionista e estatística — e reconhecer como elas coexistem nos sistemas contemporâneos;
2. **Situar** a evolução da IA em um eixo histórico que articula avanços científicos, ciclos de investimento e demandas operacionais da defesa;
3. **Distinguir**, com precisão técnica, os três regimes de aprendizado — supervisionado, não supervisionado e por reforço — e escolher o regime adequado a um problema operacional;
4. **Descrever** as cinco etapas do pipeline de um projeto de machine learning — dados, modelo, avaliação, implantação e operação;
5. **Articular** a noção de autonomia tecnológica — central à *Estratégia Nacional de Defesa* — ao caso particular da IA.

> **Como usar este notebook:** execute as células na ordem (`Shift + Enter`). Cada seção do capítulo é acompanhada de código executável que *demonstra* o conceito — não apenas o descreve. Os trechos marcados como **✏️ Experimente** convidam você a alterar parâmetros e observar o efeito.

## Preparação do ambiente

O Google Colab já traz instaladas todas as bibliotecas necessárias (`scikit-learn`, `numpy`, `matplotlib`). A célula abaixo apenas fixa as versões usadas e a semente aleatória — prática que, como veremos na etapa de **implantação** do pipeline, é requisito de reprodutibilidade em sistemas críticos.

In [ ]:
# Preparacao do ambiente: importacoes e semente de reprodutibilidade
import numpy as np
import matplotlib.pyplot as plt
import sklearn

SEMENTE = 0
np.random.seed(SEMENTE)

print(f"numpy        {np.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print("Ambiente pronto.")

---
## 1.1 As três tradições da Inteligência Artificial

A expressão *Inteligência Artificial* carrega hoje uma carga retórica que confunde o leitor desprevenido. A IA, como disciplina científica, **não é uma teoria única**: é, há mais de sessenta anos, a convivência tensa de **três tradições de pesquisa**, cada uma com uma resposta distinta para a pergunta — *o que é, computacionalmente, um comportamento inteligente?*

| Tradição | Tese central | Força | Fraqueza |
|---|---|---|---|
| **Simbólica** (GOFAI) | Inteligência é manipulação de símbolos segundo regras explícitas | Explicabilidade: cada conclusão rastreia-se à regra que a produziu | Fragilidade em percepção (dados ruidosos, ambíguos, de alta dimensionalidade) |
| **Conexionista** | Inteligência é propagação de ativações em redes de unidades elementares | Desempenho em percepção com dados em escala | Opacidade: na forma bruta, o modelo não se justifica |
| **Estatística** | Inteligência é inferência sob incerteza | Rigor na medida de incerteza: predição **+** confiança | Menos expressiva em dados brutos de altíssima dimensão |

A seguir, vemos as três tradições **em código**, resolvendo variantes do mesmo problema didático: classificar um contato detectado por um sensor.

### 1.1.1 A tradição simbólica em código

Um sistema simbólico raciocina sobre **regras explícitas**. Abaixo, um motor de regras mínimo classifica um contato naval segundo o envelope operacional — exatamente o tipo de componente que, em arquiteturas híbridas reais, **audita** as saídas dos modelos aprendidos. Note a propriedade central: cada decisão vem acompanhada da regra que a produziu.

In [ ]:
# Tradicao simbolica: motor de regras minimo para classificacao de contato
# Cada regra e um par (condicao, conclusao). A decisao e rastreavel.

REGRAS = [
    (lambda c: c["velocidade_nos"] > 35,
     "ALERTA: velocidade incompativel com trafego mercante"),
    (lambda c: c["transponder_ais"] is False and c["distancia_costa_mn"] < 12,
     "ALERTA: AIS desligado em aguas territoriais"),
    (lambda c: c["velocidade_nos"] <= 35 and c["transponder_ais"] is True,
     "ROTINA: contato compativel com trafego comercial"),
]

def classificar_contato(contato):
    """Aplica as regras em ordem e retorna (conclusao, regra_indice)."""
    for indice, (condicao, conclusao) in enumerate(REGRAS, start=1):
        if condicao(contato):
            return conclusao, indice
    return "INDETERMINADO: nenhuma regra aplicavel", None

contatos = [
    {"id": "C-01", "velocidade_nos": 42, "transponder_ais": True,  "distancia_costa_mn": 30},
    {"id": "C-02", "velocidade_nos": 14, "transponder_ais": False, "distancia_costa_mn": 8},
    {"id": "C-03", "velocidade_nos": 18, "transponder_ais": True,  "distancia_costa_mn": 50},
]

for c in contatos:
    conclusao, regra = classificar_contato(c)
    print(f"{c['id']}: {conclusao}  [justificativa: regra #{regra}]")

**Observe:** a saída inclui `[justificativa: regra #N]`. Essa é a força da tradição simbólica — **explicabilidade nativa**. E sua fraqueza também está visível: se o contato não se encaixa em nenhuma regra, o sistema devolve `INDETERMINADO`. Mundos abertos e ruidosos quebram sistemas de regras.

### 1.1.2 A tradição conexionista em código

A tradição conexionista, nascida com o **perceptron** de Frank Rosenblatt (fim dos anos 1950), parte da hipótese oposta: não há regras explícitas; há **pesos** ajustados a partir de exemplos. Abaixo, um perceptron de camada única aprende a separar duas classes — e visualizamos a fronteira de decisão que **emerge dos dados**, sem que nenhuma regra tenha sido escrita.

In [ ]:
# Tradicao conexionista: perceptron de camada unica
# Nao ha regras: ha pesos ajustados por exemplos.
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=200, centers=2, cluster_std=1.2, random_state=SEMENTE)

perceptron = Perceptron(random_state=SEMENTE)
perceptron.fit(X, y)

# Grade para desenhar a fronteira de decisao aprendida
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
Z = perceptron.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.15, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
plt.title("Perceptron: fronteira de decisao aprendida a partir de exemplos")
plt.xlabel("caracteristica 1")
plt.ylabel("caracteristica 2")
plt.show()

print("Pesos aprendidos:", perceptron.coef_.round(3))
print("Vies (bias):     ", perceptron.intercept_.round(3))

**Observe:** os "pesos aprendidos" são apenas números — não há regra legível. Em 1969, Minsky e Papert demonstraram que o perceptron de camada única não resolve problemas não linearmente separáveis (como o XOR), episódio associado ao **primeiro inverno da IA**. A célula seguinte reproduz esse limite histórico.

> **✏️ Experimente:** aumente `cluster_std` para `3.0` e re-execute. O que acontece com a fronteira quando as classes se sobrepõem?

In [ ]:
# O limite historico do perceptron: o problema XOR (Minsky & Papert, 1969)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # saida XOR: nao linearmente separavel

p = Perceptron(max_iter=1000, random_state=SEMENTE)
p.fit(X_xor, y_xor)
acuracia = p.score(X_xor, y_xor)

print(f"Acuracia do perceptron no XOR: {acuracia:.0%}")
print("Um perceptron de camada unica NAO resolve o XOR —")
print("limite formal demonstrado em 1969, gatilho do primeiro inverno da IA.")
print("A solucao (camadas ocultas + retropropagacao) sera vista no Capitulo 3.")

### 1.1.3 A tradição estatística em código

A terceira tradição compreende a inteligência como **inferência sob incerteza**. Sua força é o rigor da medida de incerteza: o modelo produz não apenas uma predição, mas **uma estimativa de quanto se pode confiar nela**. Em sistemas críticos de defesa, essa diferença não é detalhe técnico — é a fronteira entre uma ferramenta auditável e uma caixa-preta.

In [ ]:
# Tradicao estatistica: regressao logistica com probabilidades calibraveis
# A predicao vem acompanhada de uma medida de incerteza.
from sklearn.linear_model import LogisticRegression

modelo_estatistico = LogisticRegression(random_state=SEMENTE)
modelo_estatistico.fit(X, y)  # mesmos dados da secao anterior

novos_contatos = np.array([
    [X[:, 0].mean(), X[:, 1].mean()],   # ponto ambiguo, entre as classes
    [X[y == 1][:, 0].mean(), X[y == 1][:, 1].mean()],  # centro da classe 1
])

probabilidades = modelo_estatistico.predict_proba(novos_contatos)

for i, (ponto, prob) in enumerate(zip(novos_contatos, probabilidades), start=1):
    classe = prob.argmax()
    print(f"Contato {i}: classe predita = {classe} | "
          f"P(classe 0) = {prob[0]:.2f}, P(classe 1) = {prob[1]:.2f}")

print()
print("Note a diferenca: o contato ambiguo recebe probabilidades proximas de 0.5.")
print("Um sistema critico pode usar esse valor para ESCALAR a decisao a um humano.")

### 1.1.4 Como as três convivem hoje: arquiteturas híbridas

Não há, em sistemas reais, uma única tradição. O sistema de visão de uma aeronave remotamente pilotada moderna usa uma **rede convolucional profunda** (conexionista) para detectar alvos; combina as detecções com um **filtro de Kalman** (estatístico) para rastreamento; e submete as decisões finais a um **motor de regras** (simbólico) que codifica o envelope operacional autorizado.

```
  sensor ──► [CNN: detecção]──► [Kalman: rastreamento]──► [Regras: envelope]──► decisão
              conexionista        estatística               simbólica
```

> **🛡️ Contexto de defesa** — Cada tradição responde de forma distinta à exigência de **rastreabilidade**: o sistema simbólico se justifica exibindo a regra aplicada; o estatístico, exibindo a distribuição e o intervalo de confiança; a rede profunda, na forma bruta, *não se justifica* — daí o investimento em XAI e em camadas de verificação simbólica. Saber qual tradição opera em cada componente é, no limite, saber **quem responde quando o sistema erra**.

---
## 1.2 Um breve histórico, sob a ótica da defesa

Nenhum dos grandes saltos da IA aconteceu fora do contexto militar. A história da disciplina é, em larga medida, a história da convivência entre a pesquisa acadêmica, a indústria de tecnologia e o financiamento de defesa. Os marcos:

| Período | Fase | Marcos |
|---|---|---|
| 1940–1956 | Da cibernética a Dartmouth | Bletchley Park (Turing, Colossus), ENIAC (tabelas balísticas), cibernética de Wiener (mira antiaérea), Conferência de Dartmouth (1956) |
| 1956–1980 | Primeiro entusiasmo e **primeiro inverno** | GPS, SHRDLU; *Perceptrons* (Minsky & Papert, 1969); Relatório Lighthill (1973) |
| 1980–1995 | Sistemas especialistas e **segundo inverno** | MYCIN, XCON; *Fifth Generation Project* (Japão); *Strategic Computing Initiative* (DARPA) |
| 1995–2012 | Estatística, dados e maturação | SVM, boosting, random forests; a IA se dissolve em *machine learning* |
| 2012– | Revolução do aprendizado profundo | AlexNet/ImageNet (2012); Transformers; LLMs; **Project Maven** (2017) |

A lição do período é instrutiva: nos dois invernos, **a IA não fracassou tecnicamente; ela fracassou em calibrar expectativas com seus financiadores** — um padrão que se repetiria. A célula abaixo desenha a linha do tempo.

In [ ]:
# Linha do tempo da IA sob a otica da defesa
eventos = [
    (1943, "Colossus / cibernetica (2a Guerra)"),
    (1956, "Conferencia de Dartmouth"),
    (1969, "Perceptrons (Minsky & Papert)"),
    (1973, "Relatorio Lighthill -> 1o inverno"),
    (1982, "Sistemas especialistas / DARPA SCI"),
    (1990, "2o inverno"),
    (1998, "Era estatistica: SVM, ensembles"),
    (2012, "AlexNet / ImageNet"),
    (2017, "Project Maven"),
    (2023, "LLMs em OSINT e apoio a decisao"),
]

fig, ax = plt.subplots(figsize=(12, 3.5))
anos = [a for a, _ in eventos]
ax.hlines(0, 1940, 2028, color="navy", linewidth=1)
ax.plot(anos, [0] * len(anos), "o", color="navy", markersize=7)

for i, (ano, rotulo) in enumerate(eventos):
    ax.annotate(f"{ano}\n{rotulo}", (ano, 0),
                xytext=(0, 28 if i % 2 == 0 else -42),
                textcoords="offset points", ha="center", fontsize=8)

# Sombreia os dois invernos da IA
ax.axvspan(1973, 1980, alpha=0.12, color="steelblue")
ax.axvspan(1990, 1995, alpha=0.12, color="steelblue")
ax.text(1976.5, 0.035, "1o inverno", ha="center", fontsize=8, color="navy")
ax.text(1992.5, 0.035, "2o inverno", ha="center", fontsize=8, color="navy")

ax.set_ylim(-0.06, 0.06)
ax.set_xlim(1938, 2030)
ax.get_yaxis().set_visible(False)
for lado in ["left", "top", "right"]:
    ax.spines[lado].set_visible(False)
ax.set_title("A IA sob a otica da defesa: entusiasmos e invernos")
plt.tight_layout()
plt.show()

### 1.2.6 A defesa como *driver* e *consumidor*

A defesa atua, ao mesmo tempo, como **driver** (seu financiamento direcionou agendas — visão em imagens SAR, sinais em ambientes adversariais, RL em simulação de combate) e **consumidor** (adota, frequentemente com atraso por exigências de certificação, modelos consolidados primeiro no mundo civil). Boa parte das tensões do Capítulo 10 nasce do **descompasso** entre o ritmo da pesquisa, o dos requisitos operacionais e o — mais lento — da regulação.

> **🛡️ Contexto de defesa** — A *Estratégia Nacional de Defesa*, em sua ênfase na nacionalização de tecnologias sensíveis, identifica a IA como área em que a dependência externa se torna risco operacional. Adotar acriticamente modelos pré-treinados em *datasets* estrangeiros — em que a Amazônia ou a Amazônia Azul aparecem mal ou não aparecem — é **importar um viés geográfico** no produto final. Construir soberanamente significa, em IA, **controlar a cadeia que vai dos dados de coleta ao modelo implantado**.

---
## 1.3 Os três regimes de aprendizado

Aqui interessa uma distinção **operacional**: *de que forma um modelo aprende a partir dos dados que lhe são apresentados?* A escolha entre os três regimes é, talvez, **a primeira decisão estratégica** de qualquer projeto — antes de qualquer algoritmo, antes de qualquer biblioteca.

| Regime | O que se tem | O que se busca | Exemplos em defesa |
|---|---|---|---|
| **Supervisionado** | Pares rotulados $(x_i, y_i)$ | Função $\hat{f}(x) \approx y$ que generalize | Classificação de assinaturas acústicas; detecção de alvos em imagens; previsão de falha |
| **Não supervisionado** | Apenas entradas $x_i$ | Estrutura: grupos, eixos de variação, anomalias | Tráfego de rede atípico; movimentos suspeitos em fronteira; publicações coordenadas |
| **Por reforço** | Agente + ambiente + recompensas | Política que maximize recompensa acumulada | Trajetórias de plataformas autônomas; alocação dinâmica de recursos; controle de enxames |

### 1.3.1 Aprendizado supervisionado — demonstração

Simulamos um problema análogo à **classificação de assinaturas acústicas**: cada amostra tem características extraídas de um espectro, e o rótulo indica o tipo de plataforma. O gargalo prático, em projetos reais, é a **rotulagem** — produzir os pares $(x_i, y_i)$ exige especialistas (sonaristas, analistas de imagem) cujo tempo é a verdadeira matéria-prima do projeto.

In [ ]:
# Regime supervisionado: classificacao com dados rotulados
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Dados sinteticos: 2 classes de "assinatura", 6 caracteristicas espectrais
X_sup, y_sup = make_classification(n_samples=600, n_features=6,
                                   n_informative=4, random_state=SEMENTE)

X_tr, X_te, y_tr, y_te = train_test_split(X_sup, y_sup, test_size=0.25,
                                          random_state=SEMENTE, stratify=y_sup)

clf = LogisticRegression(max_iter=1000, random_state=SEMENTE)
clf.fit(X_tr, y_tr)            # aprende com os pares (x, y) de treino
y_pred = clf.predict(X_te)     # generaliza para dados nunca vistos

print(f"Amostras rotuladas de treino: {len(y_tr)}")
print(f"Acuracia em dados NUNCA vistos: {accuracy_score(y_te, y_pred):.3f}")
print()
print("O modelo so existe porque alguem rotulou 600 amostras.")
print("Em defesa, esse 'alguem' e um especialista escasso: o custo esta ai.")

### 1.3.2 Aprendizado não supervisionado — demonstração

Agora **jogamos fora os rótulos**: dispomos apenas das entradas. Usamos *k-means* para descobrir agrupamentos e, em seguida, detecção de anomalias — o caso de uso típico da fase de **vigilância de comportamento** (detectar o que destoa do padrão histórico), como tráfego de rede atípico.

In [ ]:
# Regime nao supervisionado: clustering + deteccao de anomalias, SEM rotulos
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_blobs

# "Trafego normal" em 3 padroes de comportamento + alguns pontos anomalos
X_normal, _ = make_blobs(n_samples=450, centers=3, cluster_std=1.0,
                         random_state=SEMENTE)
rng = np.random.default_rng(SEMENTE)
X_anomalo = rng.uniform(low=X_normal.min() - 4, high=X_normal.max() + 4,
                        size=(12, 2))
X_ns = np.vstack([X_normal, X_anomalo])

# 1) Descobrir estrutura: k-means encontra os 3 grupos sozinho
kmeans = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE)
grupos = kmeans.fit_predict(X_ns)

# 2) Vigilancia de comportamento: Isolation Forest sinaliza o que destoa
detector = IsolationForest(contamination=0.03, random_state=SEMENTE)
anomalia = detector.fit_predict(X_ns)  # -1 = anomalo, +1 = normal

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_ns[:, 0], X_ns[:, 1], c=grupos, cmap="viridis", s=25)
axes[0].set_title("k-means: estrutura descoberta sem nenhum rotulo")

cores = np.where(anomalia == -1, "crimson", "lightsteelblue")
axes[1].scatter(X_ns[:, 0], X_ns[:, 1], c=cores, s=25)
axes[1].set_title("Isolation Forest: pontos que destoam do padrao (vermelho)")
plt.show()

print(f"Pontos sinalizados como anomalos: {(anomalia == -1).sum()}")
print("Nenhum rotulo foi usado: a estrutura veio dos proprios dados.")

> **✏️ Experimente:** altere `n_clusters` para `2` ou `5` na célula acima. O k-means sempre encontra o número de grupos que você pedir — *saber quantos grupos existem* é, em si, um problema. Esse é um limite prático importante do regime não supervisionado.

### 1.3.3 Aprendizado por reforço — demonstração

O terceiro regime é **qualitativamente diferente**: não há conjunto de dados pré-existente; há um **agente** que interage com um **ambiente**, escolhendo ações e recebendo **recompensas**. Abaixo, um agente *Q-learning* tabular aprende, por tentativa e erro, a atravessar um pequeno *grid* com obstáculos — versão minimalista do planejamento de trajetória de uma plataforma autônoma (tema do Capítulo 7).

In [ ]:
# Regime por reforco: Q-learning tabular em um grid 5x5 com obstaculos
# O agente aprende uma POLITICA por tentativa e erro, sem dados previos.

LINHAS, COLUNAS = 5, 5
OBSTACULOS = {(1, 1), (1, 3), (3, 1), (2, 3)}
INICIO, OBJETIVO = (0, 0), (4, 4)
ACOES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # cima, baixo, esq, dir

def passo(estado, acao):
    """Aplica a acao; devolve (novo_estado, recompensa, terminou)."""
    nl, nc = estado[0] + acao[0], estado[1] + acao[1]
    if not (0 <= nl < LINHAS and 0 <= nc < COLUNAS) or (nl, nc) in OBSTACULOS:
        return estado, -5.0, False      # bateu: penalidade, fica onde esta
    if (nl, nc) == OBJETIVO:
        return (nl, nc), +100.0, True   # chegou: recompensa alta
    return (nl, nc), -1.0, False        # passo comum: custo pequeno

Q = np.zeros((LINHAS, COLUNAS, len(ACOES)))
alfa, gama, epsilon = 0.5, 0.9, 0.2
rng = np.random.default_rng(SEMENTE)
recompensas_por_episodio = []

for episodio in range(300):
    estado, total, terminou, passos = INICIO, 0.0, False, 0
    while not terminou and passos < 100:
        if rng.random() < epsilon:                      # exploracao
            a = rng.integers(len(ACOES))
        else:                                           # exploracao do melhor
            a = int(Q[estado].argmax())
        novo, r, terminou = passo(estado, ACOES[a])
        # Atualizacao Q-learning: valor da acao aproxima recompensa futura
        Q[estado][a] += alfa * (r + gama * Q[novo].max() - Q[estado][a])
        estado, total, passos = novo, total + r, passos + 1
    recompensas_por_episodio.append(total)

plt.figure(figsize=(8, 4))
media_movel = np.convolve(recompensas_por_episodio, np.ones(15) / 15, mode="valid")
plt.plot(recompensas_por_episodio, alpha=0.3, label="por episodio")
plt.plot(range(14, 300), media_movel, color="navy", label="media movel (15)")
plt.xlabel("episodio"); plt.ylabel("recompensa acumulada")
plt.title("Q-learning: o agente melhora por tentativa e erro")
plt.legend(); plt.show()

In [ ]:
# Visualizando a POLITICA aprendida: a acao preferida em cada celula do grid
SETAS = {0: "^", 1: "v", 2: "<", 3: ">"}
print("Politica aprendida (S = inicio, G = objetivo, # = obstaculo):\n")
for linha in range(LINHAS):
    celulas = []
    for coluna in range(COLUNAS):
        if (linha, coluna) in OBSTACULOS:
            celulas.append("#")
        elif (linha, coluna) == OBJETIVO:
            celulas.append("G")
        elif (linha, coluna) == INICIO:
            celulas.append("S")
        else:
            celulas.append(SETAS[int(Q[linha, coluna].argmax())])
    print("  " + "  ".join(celulas))

print()
print("A 'politica' — funcao do estado a acao — foi DESCOBERTA pelo agente.")
print("Ninguem programou a rota; ninguem rotulou dados. Esse e o regime")
print("conceitualmente mais proximo da decisao autonoma (Capitulos 7 e 10).")

> **✅ Boa prática** — A primeira decisão técnica de qualquer projeto operacional **não é a escolha do algoritmo, mas a escolha do regime de aprendizado**. Frequentemente uma equipe se compromete cedo com o supervisionado — porque é o regime mais ensinado — e descobre tarde que não tem rótulos confiáveis, ou que os rótulos foram gerados por um processo enviesado que o modelo aprenderá a reproduzir. Reservar, na fase inicial, um workshop dedicado a essa decisão — *este problema admite rótulos? Em que escala? A que custo? Com que viés sistemático?* — economiza meses no fim.

---
## 1.4 O pipeline de um projeto de machine learning

Um modelo de machine learning **é um produto**, e produtos se constroem em etapas. O pipeline de cinco etapas abaixo (variantes: CRISP-DM, CRISP-ML(Q), ciclo MLOps) organiza todo o trabalho dos capítulos seguintes:

```
 ┌────────┐   ┌────────┐   ┌───────────┐   ┌─────────────┐   ┌──────────┐
 │ 1.DADOS│──►│2.MODELO│──►│3.AVALIAÇÃO│──►│4.IMPLANTAÇÃO│──►│5.OPERAÇÃO│
 └────────┘   └────────┘   └───────────┘   └─────────────┘   └────┬─────┘
      ▲                                                           │
      └──────────────── retreinamento / drift ────────────────────┘
```

1. **Dados** — a etapa mais subestimada: fontes, ingestão, qualidade, transformações, rotulagem. Consome **60% a 80%** do tempo total; em defesa (segurança, classificação, cadeia de custódia), ainda mais.
2. **Modelo** — a que recebe atenção *desproporcional*: família, arquitetura, otimização, hiperparâmetros.
3. **Avaliação** — onde os bons projetos se separam: métricas adequadas (acurácia engana em dados desbalanceados), validação que respeite a estrutura dos dados, *baseline* honesta, análise de erros por subgrupo.
4. **Implantação** — o que separa o exercício acadêmico do produto operacional: empacotamento reproduzível, latência, telemetria (Capítulo 9).
5. **Operação** — o modelo implantado convive com *drift*, ataques deliberados e requisitos que evoluem. É a origem do **MLOps** (Capítulo 10).

### Listagem 1.1 — Esqueleto mínimo do pipeline (do livro)

A célula abaixo reproduz, na íntegra, a Listagem 1.1 do capítulo. Execute-a e confira o relatório de classificação — o F1 esperado situa-se na faixa de **85% a 90%**.

In [ ]:
# Pipeline minimo de machine learning supervisionado.
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Dados ------------------------------------------------------------
X, y = make_classification(n_samples=1000, n_features=8, random_state=0)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# 2. Modelo (com transformacao de dados acoplada) ---------------------
escalador = StandardScaler()
X_treino_n = escalador.fit_transform(X_treino)
X_teste_n = escalador.transform(X_teste)  # mesma transformacao do treino

modelo = LogisticRegression(max_iter=1000, random_state=0)
modelo.fit(X_treino_n, y_treino)

# 3. Avaliacao --------------------------------------------------------
y_predito = modelo.predict(X_teste_n)
print(classification_report(y_teste, y_predito, digits=3))

# 4. Implantacao (esboco) ---------------------------------------------
# import joblib
# joblib.dump({"escalador": escalador, "modelo": modelo}, "modelo_v1.joblib")

# 5. Operacao (esboco) ------------------------------------------------
# - monitorar a distribuicao das entradas em producao
# - recolher feedback de erros para retreinamento periodico
# - re-validar antes de re-implantar

**Mais importante que o número é a estrutura:** as cinco etapas estão lá, nomeadas, com a lógica que revisitaremos em todos os modelos seguintes. Repare em um detalhe da etapa 2: `fit_transform` no **treino** e apenas `transform` no **teste**. Esse detalhe é a diferença entre um pipeline correto e a armadilha mais perigosa da área — demonstrada a seguir.

### ⚠️ Armadilha comum: fuga de informação (*data leakage*)

*Data leakage* ocorre quando algum sinal sobre a resposta **vaza**, durante a preparação dos dados, para o conjunto de treinamento. O modelo atinge métricas excelentes na validação e **falha em produção**, onde o sinal vazado não existe. A célula abaixo demonstra a forma mais sutil: padronizar o conjunto **inteiro** antes de separar treino e teste.

In [ ]:
# Demonstracao de data leakage: padronizar ANTES de separar treino/teste
# O vazamento aqui e sutil: as estatisticas (media, desvio) do TESTE
# contaminam a transformacao aplicada ao TREINO.
from sklearn.model_selection import cross_val_score

X_leak, y_leak = make_classification(n_samples=300, n_features=20,
                                     n_informative=2, n_redundant=18,
                                     flip_y=0.3, random_state=SEMENTE)

# --- Forma ERRADA: escala o conjunto inteiro, depois separa ----------
escalador_errado = StandardScaler().fit(X_leak)       # ve TODOS os dados
X_todo_escalado = escalador_errado.transform(X_leak)
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_todo_escalado, y_leak, test_size=0.3, random_state=SEMENTE)
m_errado = LogisticRegression(max_iter=1000).fit(Xa_tr, ya_tr)
acc_errada = m_errado.score(Xa_te, ya_te)

# --- Forma CORRETA: separa primeiro, escala com estatisticas do treino
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X_leak, y_leak, test_size=0.3, random_state=SEMENTE)
esc = StandardScaler().fit(Xb_tr)                     # ve SO o treino
m_correto = LogisticRegression(max_iter=1000).fit(esc.transform(Xb_tr), yb_tr)
acc_correta = m_correto.score(esc.transform(Xb_te), yb_te)

print(f"Acuracia com vazamento (errada) : {acc_errada:.3f}")
print(f"Acuracia sem vazamento (correta): {acc_correta:.3f}")
print()
print("Neste exemplo didatico a diferenca e pequena; em series temporais")
print("e cruzamento de bases de sistemas distintos — caso tipico em defesa —")
print("o vazamento pode inflar metricas em dezenas de pontos percentuais.")
print("Antes de comemorar uma metrica alta, pergunte:")
print("COMO, exatamente, este modelo aprendeu a acertar tanto?")

> **🛡️ Da prova de conceito ao sistema crítico** — Em defesa, **nenhum modelo é entregue como prova de conceito**. Mesmo ferramentas de apoio à análise — que apenas sugerem hipóteses ao analista — precisam ser auditáveis, reproduzíveis e robustas a entradas mal-intencionadas. A passagem da prova de conceito ao sistema crítico é a que custa anos — e é o tema central dos Módulos III e IV do livro.

---
## 1.5 O caminho à frente

| Módulo | Capítulos | Tema |
|---|---|---|
| **I — Fundamentos e Modelos Clássicos** | 1–3 | Modelos supervisionados clássicos; o salto para o deep learning |
| **II — Percepção: Visão e Linguagem** | 4–6 | Visão computacional (YOLO, Faster R-CNN, SAR); PLN para inteligência; LLMs em OSINT |
| **III — Decisão Autônoma e RL** | 7–8 | RL profundo para decisão tática; robustez adversarial |
| **IV — Confiabilidade, Operação e Governança** | 9–10 | IA embarcada e tempo real; quadro ético-regulatório e controle humano significativo |

---

### 📋 Resumo do capítulo

- A IA não é uma teoria única, mas a convivência de **três tradições** — simbólica, conexionista e estatística. Sistemas reais combinam as três em **arquiteturas híbridas**.
- A história da disciplina, sob a ótica da defesa, é uma sequência de **entusiasmos e invernos** produzida pelo descompasso entre promessas e expectativas. O ponto de inflexão contemporâneo é a revolução do aprendizado profundo (2012–), com agenda de defesa explícita a partir do **Project Maven** (2017).
- Três **regimes de aprendizado** organizam a prática: supervisionado, não supervisionado e por reforço. A escolha do regime adequado é **a primeira decisão estratégica** do projeto.
- O **pipeline de cinco etapas** — dados, modelo, avaliação, implantação, operação — organiza todo o livro. Em sistemas críticos, o problema central é a passagem da prova de conceito ao sistema implantado.
- A **autonomia tecnológica** se traduz, em IA, em controlar a cadeia que vai dos dados de coleta ao modelo implantado. Adotar acriticamente modelos estrangeiros importa, junto, seus vieses.

### ⚠️ Armadilhas comuns

1. **Confundir IA com deep learning.** Em muitas aplicações, modelos clássicos são mais auditáveis, mais robustos e mais adequados.
2. **Subestimar a fase de dados.** Em projetos reais, o gargalo está quase sempre na qualidade, rotulagem e curadoria dos dados.
3. **Tratar a avaliação como uma única métrica.** Acurácia agregada em conjuntos desbalanceados engana; é preciso métrica diferenciada por tipo de erro e análise por subgrupo.
4. **Confundir notebook com sistema implantado.** Versões fixadas, latência, telemetria, monitoramento de *drift* — tudo se torna visível só na transição para operação.
5. **Importar viés geográfico via modelos pré-treinados** sem avaliar o desempenho em dados nacionais.

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código (1.3 e 1.6) têm células preparadas abaixo; os demais são dissertativos — use células de texto para respondê-los no próprio notebook, se desejar.

### Essencial

**Exercício 1.1** — Em até uma página, contraste as três tradições da IA, indicando para cada uma: (a) sua tese central sobre o que é comportamento inteligente; (b) um exemplo de aplicação em que ela é, ainda hoje, a melhor escolha técnica; (c) uma limitação operacional típica.

**Exercício 1.2** — Identifique, em uma reportagem técnica recente sobre IA em defesa, qual das três tradições parece operar no sistema descrito. Formule duas perguntas que você faria à equipe responsável para verificar sua hipótese.

**Exercício 1.3** — Execute o esqueleto do pipeline (Listagem 1.1, acima). Em seguida, na célula abaixo, altere `test_size` para `0.50` e relate, em duas linhas, o efeito sobre as métricas.

In [ ]:
# Exercicio 1.3 — altere test_size e compare com o resultado original
X, y = make_classification(n_samples=1000, n_features=8, random_state=0)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.25,   # <--- ALTERE AQUI para 0.50 e re-execute
    random_state=0, stratify=y
)

escalador = StandardScaler()
X_treino_n = escalador.fit_transform(X_treino)
X_teste_n = escalador.transform(X_teste)

modelo = LogisticRegression(max_iter=1000, random_state=0)
modelo.fit(X_treino_n, y_treino)
print(classification_report(y_teste, modelo.predict(X_teste_n), digits=3))

# Sua analise (duas linhas):
# 1) ...
# 2) ...

### Tático

**Exercício 1.4** — Para o problema de detecção de embarcações suspeitas a partir de dados AIS, liste **três decisões técnicas concretas** para cada uma das quatro etapas iniciais do pipeline (dados, modelo, avaliação, implantação) e identifique, em cada decisão, o **principal risco operacional** associado.

**Exercício 1.5** — Sob qual regime de aprendizado você atacaria os problemas abaixo? Justifique. (a) prever consumo de combustível de uma frota a partir do perfil de missão; (b) descobrir grupos de comportamento atípico em metadados de tráfego de uma rede tática; (c) treinar um agente para conduzir um VANT em ambiente simulado com obstáculos móveis.

**Exercício 1.6** — Substitua, na célula abaixo, `LogisticRegression` por `RandomForestClassifier`. Compare os relatórios. Em uma linha, formule uma hipótese sobre por que a diferença ocorre; na linha seguinte, descreva um experimento que permitiria testá-la.

In [ ]:
# Exercicio 1.6 — troque o classificador e compare
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(n_samples=1000, n_features=8, random_state=0)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

escalador = StandardScaler()
X_treino_n = escalador.fit_transform(X_treino)
X_teste_n = escalador.transform(X_teste)

# Modelo original:
modelo = LogisticRegression(max_iter=1000, random_state=0)
# Substitua pela linha abaixo e re-execute:
# modelo = RandomForestClassifier(random_state=0)

modelo.fit(X_treino_n, y_treino)
print(type(modelo).__name__)
print(classification_report(y_teste, modelo.predict(X_teste_n), digits=3))

# Hipotese (1 linha): ...
# Experimento para testa-la (1 linha): ...

### Estratégico

**Exercício 1.7** — Leia o trecho da *Estratégia Nacional de Defesa* que trata da nacionalização de tecnologias sensíveis. Em um breve ensaio (duas páginas), discuta como o pipeline de machine learning se articula ao princípio da **soberania tecnológica** — em que pontos da cadeia, especificamente, a dependência externa se torna risco operacional.

**Exercício 1.8** — Esboce, em um diagrama, a arquitetura híbrida de um sistema operacional de sua escolha (por exemplo, vigilância marítima), explicitando qual tradição da IA opera em cada componente. Indique em que pontos as interfaces entre componentes representam risco — de *drift*, de ataque adversarial, de incompatibilidade de garantias.

**Exercício 1.9** — Escolha um dataset brasileiro de domínio público (INPE, IBGE, iniciativas de governo aberto). Avalie, em duas páginas, se ele seria adequado como base para algum dos problemas operacionais deste capítulo — identificando vieses sistemáticos, cobertura geográfica e limites de uso documental.

---

### 📚 Referências principais do capítulo

- RUSSELL, S.; NORVIG, P. *Artificial Intelligence: A Modern Approach*.
- GOODFELLOW, I.; BENGIO, Y.; COURVILLE, A. *Deep Learning*.
- GÉRON, A. *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*.
- ALPAYDIN, E. *Introduction to Machine Learning*.
- BRASIL. *Estratégia Nacional de Defesa*.

*Próximo passo: Capítulo 2 — Modelos Clássicos de Aprendizado Supervisionado.*